In [1]:
# Install required libraries
!pip install tensorflow keras opencv-python kaggle

In [2]:
import os

os.environ['KAGGLE_USERNAME'] = "mranju"
os.environ['KAGGLE_KEY'] = "KGAT_3aa12a868e44af1a41918854e1f463bb"

In [3]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [6]:
!kaggle datasets download -d fanconic/skin-cancer-malignant-vs-benign

Dataset URL: https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign
License(s): unknown
 98% 319M/325M [00:02<00:00, 89.8MB/s]
100% 325M/325M [00:02<00:00, 114MB/s] 


In [7]:
import zipfile

with zipfile.ZipFile("skin-cancer-malignant-vs-benign.zip", "r") as zip_ref:
    zip_ref.extractall("skin_cancer_data")

In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image generator with normalization and augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% of train images for validation
)

# Train generator
train_generator = train_datagen.flow_from_directory(
    "skin_cancer_data/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

# Validation generator
val_generator = train_datagen.flow_from_directory(
    "skin_cancer_data/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,   # You can increase to 20-30 if needed
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.6346 - loss: 0.6303 - val_accuracy: 0.7514 - val_loss: 0.5230
Epoch 2/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.7605 - loss: 0.4809 - val_accuracy: 0.7590 - val_loss: 0.4864
Epoch 3/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - accuracy: 0.7942 - loss: 0.4251 - val_accuracy: 0.7021 - val_loss: 0.5281
Epoch 4/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.8083 - loss: 0.4092 - val_accuracy: 0.7457 - val_loss: 0.4801
Epoch 5/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.8088 - loss: 0.3850 - val_accuracy: 0.7685 - val_loss: 0.4924
Epoch 6/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8153 - loss: 0.4066 - val_accuracy: 0.7913 - val_loss: 0.4880
Epoch 7/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step - accuracy: 0.8064 - loss: 0.3796 - val_accuracy: 0.7666 - val_loss: 0.4782
Epoch 8/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.8268 - loss: 0.3665 - val_accuracy: 0.8008 - val_loss:

In [11]:
model.save("skin_cancer_cnn.h5")
print("Model saved as skin_cancer_cnn.h5")

Model saved as skin_cancer_cnn.h5


In [12]:
from tensorflow.keras.preprocessing import image
import numpy as np
from google.colab import files

# Upload a new image to test
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load and preprocess the image
img = image.load_img(filename, target_size=(128,128))
img_arr = image.img_to_array(img)/255.0
img_arr = np.expand_dims(img_arr, axis=0)

# Predict
pred = model.predict(img_arr)
if pred[0][0] > 0.5:
    print("Prediction: Malignant")
else:
    print("Prediction: Benign")

Saving 1003.jpg to 1003.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Prediction: Benign
